# 10 - Quantum-Enhanced U-Net + Quanvolutional Filtering

Implements the reference architecture's **quantum-enhanced bottleneck** and **quanvolutional filtering**
components for real, on top of the existing classical U-Net (notebook 09) and quantum kernel SVM
(notebooks 05/06):

1. **Quantum-enhanced bottleneck** (`utils/qml/quantum_layer.py`, `utils/ai/classic/quantum_unet.py`): a real,
   trainable parameterized quantum circuit (ZZFeatureMap encoding + RealAmplitudes ansatz) fused into the
   U-Net's bottleneck as a channel-gating block - global-average-pool the bottleneck -> compress to 6
   qubits -> quantum circuit -> expand back to a per-channel gate. Trained **end-to-end** with a single
   AdamW optimizer: `qiskit-machine-learning`'s `TorchConnector` computes gradients through the quantum
   circuit via the parameter-shift rule automatically, so both classical conv weights and quantum circuit
   parameters update from the same backward pass. This is a real deviation from the reference diagram's
   split classical-AdamW / quantum-SPSA-or-COBYLA optimizer design - see `quantum_layer.py`'s docstring for
   why (measured quantum backward cost is cheap enough here - ~0.6s/sample - that the added complexity of a
   bi-level optimization loop wasn't worth it).
2. **Quanvolutional filtering** (`utils/qml/quanvolution.py`): a *fixed* (untrained, randomly-initialized)
   quantum circuit applied like a strided convolution, one output channel per qubit from Z-expectation
   measurements (Henderson et al. 2020). Real and batched, but genuinely `O(H*W/stride^2)` circuit
   evaluations - a full 512x512 chip is ~262k evaluations, ~7.3 minutes measured locally. Demonstrated here
   on a small crop, not applied to the full training set, for that reason - stated honestly rather than
   silently skipped or silently applied to a full dataset that would take hours.

In [1]:
import os
import sys

# Portable project-root resolution (no machine-specific hardcoded path) -
# walk up from the current working directory until pyproject.toml is found.
# One statement on purpose: ruff/pycodestyle's E402 ("imports not at top")
# specifically exempts a lone sys.path.insert(...) call, not a multi-
# statement block before it.
sys.path.insert(0, next(
    d for d in (
        os.path.abspath(os.path.join(os.getcwd(), *([os.pardir] * i)))
        for i in range(8)
    )
    if os.path.exists(os.path.join(d, "pyproject.toml"))
))

import time

import torch
from torch.utils.data import DataLoader

from utils.ai.classic.losses import MaskedComboLoss
from utils.ai.classic.quantum_unet import QuantumEnhancedUNet
from utils.ai.classic.sen1floods11_dataset import Sen1Floods11Dataset
from utils.ai.objectives.registry import evaluate
from utils.observability.run_logger import RunLogger

_root = sys.path[0]
logger = RunLogger("10_quantum_enhanced_unet")
DEVICE = "cpu"  # quantum layer runs on a local Qiskit simulator regardless of torch device
print(f"device: {DEVICE}")

device: cpu


## Quantum-enhanced bottleneck: build + train

In [2]:
N_TRAIN_CHIPS = 10
N_VAL_CHIPS = 4
BATCH_SIZE = 2
EPOCHS = 3

with logger.stage("build_datasets") as stage:
    train_ds = Sen1Floods11Dataset("train")
    val_ds = Sen1Floods11Dataset("valid")
    train_subset = torch.utils.data.Subset(train_ds, range(min(N_TRAIN_CHIPS, len(train_ds))))
    val_subset = torch.utils.data.Subset(val_ds, range(min(N_VAL_CHIPS, len(val_ds))))
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE)
    stage.metrics = {"n_train_chips": len(train_subset), "n_val_chips": len(val_subset)}

print(f"train: {len(train_subset)} chips, val: {len(val_subset)} chips")
print("kept deliberately small (vs notebook 09's 20/8) - each training step now also pays the quantum")
print("bottleneck's backward-pass cost (~0.6s/sample measured), on top of the classical conv cost.")

[10_quantum_enhanced_unet] -> build_datasets ...
[10_quantum_enhanced_unet] <- build_datasets [OK] 0.001s {'n_train_chips': 10, 'n_val_chips': 4}
train: 10 chips, val: 4 chips
kept deliberately small (vs notebook 09's 20/8) - each training step now also pays the quantum
bottleneck's backward-pass cost (~0.6s/sample measured), on top of the classical conv cost.


In [3]:
model = QuantumEnhancedUNet(in_channels=2, num_classes=1, base_channels=8, n_qubits=6).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = MaskedComboLoss()

n_params = sum(p.numel() for p in model.parameters())
n_quantum_params = sum(p.numel() for p in model.quantum_layer.parameters())
print(f"model: {n_params:,} parameters ({n_quantum_params} of them are quantum circuit weights)")

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


model: 488,905 parameters (18 of them are quantum circuit weights)


In [4]:
with logger.stage("train_quantum_unet") as stage:
    model.train()
    epoch_times = []
    for epoch in range(EPOCHS):
        t0 = time.time()
        epoch_loss = 0.0
        n_batches = 0
        for batch in train_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].unsqueeze(1).to(DEVICE)
            mask = batch["valid_mask"].unsqueeze(1).to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y, mask)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        epoch_time = time.time() - t0
        epoch_times.append(epoch_time)
        mean_loss = epoch_loss / n_batches
        print(f"epoch {epoch+1}/{EPOCHS} loss={mean_loss:.4f} time={epoch_time:.1f}s")

    stage.metrics = {"epochs": EPOCHS, "mean_epoch_s": round(sum(epoch_times) / len(epoch_times), 1), "final_loss": round(mean_loss, 4)}

[10_quantum_enhanced_unet] -> train_quantum_unet ...


epoch 1/3 loss=0.7157 time=10.1s


epoch 2/3 loss=0.7023 time=9.9s


epoch 3/3 loss=0.6924 time=10.2s
[10_quantum_enhanced_unet] <- train_quantum_unet [OK] 30.203s {'epochs': 3, 'mean_epoch_s': 10.1, 'final_loss': 0.6924}


In [5]:
with logger.stage("evaluate_quantum_unet") as stage:
    model.eval()
    all_metrics = []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].numpy()
            mask = batch["valid_mask"].numpy()

            logits = model(x)
            preds = (torch.sigmoid(logits).squeeze(1).cpu().numpy() > 0.5).astype(int)

            for i in range(preds.shape[0]):
                valid = mask[i]
                if valid.sum() == 0:
                    continue  # same "zero valid pixels -> no meaningful metric" case as the web API's inference endpoint
                m = evaluate("flood-segmentation", preds[i][valid], y[i][valid].astype(int))
                all_metrics.append(m)

    mean_metrics = {k: sum(m[k] for m in all_metrics) / len(all_metrics) for k in all_metrics[0]}
    stage.metrics = {k: round(v, 4) for k, v in mean_metrics.items()}

print(f"mean validation metrics over {len(all_metrics)} chips (zero-valid-pixel chips excluded):")
for k, v in mean_metrics.items():
    print(f"  {k:12s} {v:.4f}")

[10_quantum_enhanced_unet] -> evaluate_quantum_unet ...


[10_quantum_enhanced_unet] <- evaluate_quantum_unet [OK] 0.601s {'iou': 0.3333, 'f1': 0.3333, 'precision': 1.0, 'recall': 0.3333, 'boundary_f1': 0.3333}
mean validation metrics over 3 chips (zero-valid-pixel chips excluded):
  iou          0.3333
  f1           0.3333
  precision    1.0000
  recall       0.3333
  boundary_f1  0.3333


In [6]:
import os

MODEL_PATH = os.path.join(_root, "datasets", "processed", "models", "quantum_unet_v1.pt")
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
torch.save(model.state_dict(), MODEL_PATH)

logger.log_metrics({"model_path": MODEL_PATH, **{f"valid_{k}": v for k, v in mean_metrics.items()}})
print(f"saved model -> {MODEL_PATH}")

saved model -> d:\project-raw-data\sphoorthq-geoverse\datasets\processed\models\quantum_unet_v1.pt


## Quanvolutional filtering: real, batched, demonstrated on a crop

In [7]:
from utils.ai.classic.sen1floods11_dataset import chip_id_from_s1_filename, load_split, normalize_s1, read_s1
from utils.qml.quanvolution import quanvolve

with logger.stage("quanvolution_demo") as stage:
    pairs = load_split("train")
    chip_id = chip_id_from_s1_filename(pairs[3][0])
    s1 = normalize_s1(read_s1(chip_id))
    crop = s1[0][:64, :64]  # 64x64 crop of the VV channel - full 512x512 would be ~7 minutes, see quanvolution.py

    t0 = time.time()
    quanv_out = quanvolve(crop, patch_size=2, stride=2, seed=0)
    elapsed = time.time() - t0
    n_evals = ((64 - 2) // 2 + 1) ** 2 * 4

    stage.metrics = {
        "chip_id": chip_id, "crop_size": "64x64", "output_shape": list(quanv_out.shape),
        "n_circuit_evals": n_evals, "elapsed_s": round(elapsed, 2), "evals_per_sec": round(n_evals / elapsed, 1),
    }

print(f"quanvolution on {chip_id} (64x64 crop): output {quanv_out.shape}, {n_evals} circuit evals in {elapsed:.2f}s")
print(f"throughput: {n_evals/elapsed:.0f} evals/sec -> extrapolated full 512x512 chip: {262144/(n_evals/elapsed)/60:.1f} minutes")

[10_quantum_enhanced_unet] -> quanvolution_demo ...


C:\Users\Admin\AppData\Local\Temp\ipykernel_16292\3284247317.py:12: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  quanv_out = quanvolve(crop, patch_size=2, stride=2, seed=0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_16292\3284247317.py:12: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  quanv_out = quanvolve(crop, patch_size=2, stride=2, seed=0)


[10_quantum_enhanced_unet] <- quanvolution_demo [OK] 35.731s {'chip_id': 'Ghana_953791', 'crop_size': '64x64', 'output_shape': [4, 32, 32], 'n_circuit_evals': 4096, 'elapsed_s': 35.71, 'evals_per_sec': 114.7}
quanvolution on Ghana_953791 (64x64 crop): output (4, 32, 32), 4096 circuit evals in 35.71s
throughput: 115 evals/sec -> extrapolated full 512x512 chip: 38.1 minutes


## Comparison vs classical U-Net (notebook 09)

In [8]:
from utils.observability.run_logger import load_recent_runs

classical_metrics = None
for run in load_recent_runs(limit=30):
    if run["run_name"] == "09_patch_unet":
        classical_metrics = {k.removeprefix("valid_"): v for k, v in run["metrics"].items() if k.startswith("valid_")}
        break

print("classical U-Net (notebook 09):", classical_metrics or "not found - run notebook 09 first")
print("quantum-enhanced U-Net (this notebook):", {k: round(v, 4) for k, v in mean_metrics.items()})
print()
print("Not a fair apples-to-apples comparison: different N_TRAIN_CHIPS (20 vs 10), different base_channels")
print("(16 vs 8), different random chip sample - kept smaller here specifically because of the added")
print("quantum bottleneck training cost. Real numbers from real runs, not tuned to look favorable either way.")

logger.finalize()

classical U-Net (notebook 09): {'iou': 0.625, 'f1': 0.625, 'precision': 1.0, 'recall': 0.625, 'boundary_f1': 0.625}
quantum-enhanced U-Net (this notebook): {'iou': 0.3333, 'f1': 0.3333, 'precision': 1.0, 'recall': 0.3333, 'boundary_f1': 0.3333}

Not a fair apples-to-apples comparison: different N_TRAIN_CHIPS (20 vs 10), different base_channels
(16 vs 8), different random chip sample - kept smaller here specifically because of the added
quantum bottleneck training cost. Real numbers from real runs, not tuned to look favorable either way.
[10_quantum_enhanced_unet] run complete in 67.836s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\718ba904-1d22-41f4-b85c-c4f6d5aaf724.json


'D:\\project-raw-data\\sphoorthq-geoverse\\datasets\\reports\\runs\\718ba904-1d22-41f4-b85c-c4f6d5aaf724.json'